# Compare labeling methods and classifiers using 150 embeddings

Keep the original inputs, submitted models, and Kaggle reference 0.87393 unchanged. This notebook compares methods before adding more features: **no year, extra PCA, text, or drug features**.

Stage 1 compares K-means and GMM (diagonal or tied covariance) at K=2,5,8,10,15 using C=1000 logistic regression. All new methods use the same improved label rule: assign each reason independently, retain uncertain targets as -100, and cap generated-label influence so hundreds of generated negatives cannot overwhelm a rare reason. Known labels never change.

Stage 2 shortlists the best dataset from each clustering family, plus known-only and legacy K=2 labels, then compares logistic regression C=100,1000,10000 and distance/sample-weighted nearest neighbors k=1,3,7,15. This is a staged search, not an exhaustive search of every combination.

Use the same 78 validation trials, sample-wise F1 as the primary metric, and threshold 0.5 with the nonempty fallback. Validation/test rows never participate in clustering, label estimation, scaling, or training. All new classifier comparisons share the submitted supervised baseline's scaler fitted on the 157 known-label trials. Clustering uses its separate scaler fitted on 1,171 training embeddings. Validation selects among the configurations; candidate Kaggle scores remain unknown until manual submission.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from threadpoolctl import threadpool_limits

SEED = 42
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
FEATURES = [f"emb_{i}" for i in range(150)]
K_VALUES = [2, 5, 8, 10, 15]
LABEL_RULE = {"minimum_support": 3.0, "prior_strength": 2.0,
              "positive_threshold": 0.6, "negative_threshold": 0.1,
              "maximum_generated_weight": 0.25, "maximum_class_fraction": 0.5}
GMM_RULE = {"n_init": 3, "max_iter": 500, "tol": 1e-3,
            "reg_covar": 1e-3, "random_state": SEED, "init_params": "kmeans"}
CLASSIFIER_GRID = [("logistic", c) for c in [100.0, 1000.0, 10000.0]] + [
    ("weighted_knn", k) for k in [1, 3, 7, 15]
]
cwd = Path.cwd()
DATA = next(q for q in [cwd, cwd/"part1", cwd/"AML-Midterm"/"part1"]
            if (q/"part1_train.csv").is_file() or (q/"data/part1_train.csv").is_file()).resolve()
INPUT_DIR = DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
OUT = DATA/"experiments"/"method_comparison"
for folder in ["datasets", "label_audits", "clusters", "clustering_models", "classifier_models", "validation_predictions", "submissions"]:
    (OUT/folder).mkdir(parents=True, exist_ok=True)
train = pd.read_csv(INPUT_DIR/"part1_train.csv")
valid = pd.read_csv(INPUT_DIR/"part1_val.csv")
X = train[FEATURES].to_numpy()
X_valid = valid[FEATURES].to_numpy()
truth = valid[REASONS].to_numpy(dtype=int)
original_y = train[REASONS].to_numpy(dtype=int)
known = train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert known.sum()==157 and len(train)==1171 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isfinite(X).all() and np.isfinite(X_valid).all()
assert np.isin(truth,[0,1]).all()
baseline = joblib.load(DATA/"experiments/kmeans/models/baseline_C1000.joblib")
legacy_packet = joblib.load(DATA/"experiments/kmeans/models/k02.joblib")
classifier_scaler = baseline.named_steps["scaler"]
cluster_scaler = legacy_packet["kmeans_scaler"]
assert classifier_scaler.n_samples_seen_==157 and cluster_scaler.n_samples_seen_==1171
XS = classifier_scaler.transform(X)
VS = classifier_scaler.transform(X_valid)
XC = cluster_scaler.transform(X)
protected_paths = [INPUT_DIR/"part1_train.csv", INPUT_DIR/"part1_val.csv"] + sorted(DATA.glob("*submission.csv")) + [
    DATA/"experiments/kmeans/models/k02.joblib", DATA/"experiments/kmeans/datasets/k02_train.csv"
]
protected_hashes = {str(q.relative_to(DATA)):hashlib.sha256(q.read_bytes()).hexdigest() for q in protected_paths}
print("Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.")

Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.


## Partial labels and influence budgets

For a component, estimate each reason's prevalence from known labels, with smoothing strength 2 toward known-training prevalence. K-means gives hard memberships; GMM supplies soft responsibilities. Mix component prevalences according to each trial's memberships.

Assign a positive only at prevalence >=0.6 and a negative only at prevalence <=0.1, with at least 3 effective labeled members. Other targets remain unknown. For each reason, require both positive and negative generated candidates before adding either class. Limit each generated example to weight 0.25 and each class's total generated weight to at most half the corresponding known class's weight. Use the same added fraction for both classes. This preserves the known class balance as an influence control, **not as an estimate of population prevalence**. A reason with no generated positive candidates gets no generated negatives either.

For logistic regression, normalize each reason's total fitting weight to 157 so changes in sample count do not silently weaken regularization at fixed C. Nearest-neighbor votes incorporate both distance and training sample weights. All original known labels are retained with raw weight 1.

In [2]:
def label_predictions(probabilities):
    probabilities = np.asarray(probabilities)
    assert probabilities.shape[1]==5 and np.isfinite(probabilities).all()
    assert ((probabilities>=0)&(probabilities<=1)).all()
    prediction=(probabilities>=0.5).astype(int)
    empty=prediction.sum(axis=1)==0
    ids=np.flatnonzero(empty)
    prediction[ids, probabilities[ids].argmax(axis=1)]=1
    return prediction,empty


def score_probabilities(probabilities):
    predicted,empty=label_predictions(probabilities)
    return {"sample_f1":f1_score(truth,predicted,average="samples",zero_division=0),
            "macro_f1":f1_score(truth,predicted,average="macro",zero_division=0),
            "fallback_trials":int(empty.sum()),
            **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0)))},predicted,empty


def save_predictions(name, probabilities):
    metrics,predicted,empty=score_probabilities(probabilities)
    output=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):
        output[f"probability_{r}"]=probabilities[:,j]
        output[f"predicted_{r}"]=predicted[:,j]
    output["used_fallback"]=empty
    output.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    return metrics


DATASETS={}
LABEL_COUNTS=[]
FIT_CACHE={}


def save_dataset(name, targets, weights, probabilities=None, component=None, support=None):
    assert targets.shape==weights.shape==(len(train),5)
    np.testing.assert_array_equal(targets[known],original_y[known])
    np.testing.assert_array_equal(weights[known],np.ones((known.sum(),5)))
    assert np.isin(targets,[-100,0,1]).all() and (weights>=0).all()
    assert np.array_equal(weights>0,targets!=-100)
    frame=train[["ID"]+FEATURES].copy()
    audit=train[["ID"]].copy()
    for j,r in enumerate(REASONS):
        source=np.where(known,"original",np.where(weights[:,j]>0,"generated","unknown"))
        frame[r]=targets[:,j]
        frame[f"weight_{r}"]=weights[:,j]
        audit[r]=targets[:,j]
        audit[f"original_{r}"]=original_y[:,j]
        audit[f"weight_{r}"]=weights[:,j]
        audit[f"label_source_{r}"]=source
        if probabilities is not None:audit[f"estimated_probability_{r}"]=probabilities[:,j]
        pos=(~known)&(targets[:,j]==1)
        neg=(~known)&(targets[:,j]==0)
        LABEL_COUNTS.append({"dataset":name,"reason":r,
            "known_positives":int(original_y[known,j].sum()),
            "known_negatives":int((original_y[known,j]==0).sum()),
            "generated_positives":int(pos.sum()),"generated_negatives":int(neg.sum()),
            "generated_positive_weight":float(weights[pos,j].sum()),
            "generated_negative_weight":float(weights[neg,j].sum())})
    if component is not None:audit["component"]=component
    if support is not None:audit["effective_labeled_support"]=support
    frame["included_for_any_reason"]=(weights>0).any(axis=1)
    frame.to_csv(OUT/"datasets"/f"{name}.csv",index=False)
    audit.to_csv(OUT/"label_audits"/f"{name}.csv",index=False)
    DATASETS[name]={"targets":targets.copy(),"weights":weights.copy(),
                    "generated_trials":int(((weights>0).any(axis=1)&~known).sum())}
    return DATASETS[name]


def partial_labels(responsibilities):
    assert responsibilities.shape[0]==len(train)
    np.testing.assert_allclose(responsibilities.sum(axis=1),1,atol=1e-9)
    component_support=responsibilities[known].sum(axis=0)
    positives=responsibilities[known].T@original_y[known]
    prior=original_y[known].mean(axis=0)
    rate=(positives+LABEL_RULE["prior_strength"]*prior)/(component_support[:,None]+LABEL_RULE["prior_strength"])
    probabilities=responsibilities@rate
    support=responsibilities@component_support
    targets=np.full(original_y.shape,-100,dtype=int)
    weights=np.zeros(original_y.shape,dtype=float)
    targets[known]=original_y[known]
    weights[known]=1.0
    eligible=(~known)&(support>=LABEL_RULE["minimum_support"])
    for j in range(5):
        positive=eligible&(probabilities[:,j]>=LABEL_RULE["positive_threshold"])
        negative=eligible&(probabilities[:,j]<=LABEL_RULE["negative_threshold"])
        npos,nneg=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        if not npos or not nneg:continue
        fraction=min(LABEL_RULE["maximum_class_fraction"],
                     LABEL_RULE["maximum_generated_weight"]*npos/kp,
                     LABEL_RULE["maximum_generated_weight"]*nneg/kn)
        targets[positive,j]=1;targets[negative,j]=0
        weights[positive,j]=fraction*kp/npos
        weights[negative,j]=fraction*kn/nneg
        assert weights[~known,j].max()<=LABEL_RULE["maximum_generated_weight"]+1e-12
        np.testing.assert_allclose(weights[positive,j].sum()/kp,weights[negative,j].sum()/kn,atol=1e-12)
    components=pd.DataFrame({"component":np.arange(len(component_support)),"effective_labeled_support":component_support,
                             "total_membership_mass":responsibilities.sum(axis=0)})
    for j,r in enumerate(REASONS):
        components[f"weighted_known_positive_{r}"]=positives[:,j]
        components[f"estimated_probability_{r}"]=rate[:,j]
    return targets,weights,probabilities,support,components


def fit_reason_models(dataset_name, kind="logistic", parameter=1000.0):
    dataset=DATASETS[dataset_name]
    estimators=[]
    for j in range(5):
        y=dataset["targets"][:,j];w=dataset["weights"][:,j]
        included=w>0
        assert np.isin(y[included],[0,1]).all() and included[known].all()
        key=(kind,float(parameter),j,hashlib.sha256(y.tobytes()+w.tobytes()).hexdigest())
        if key not in FIT_CACHE:
            if kind=="logistic":
                model=LogisticRegression(C=float(parameter),solver="lbfgs",max_iter=5000,tol=1e-4,random_state=SEED)
                normalized=w[included]*known.sum()/w[included].sum()
                with warnings.catch_warnings():
                    warnings.simplefilter("error",ConvergenceWarning)
                    model.fit(XS[included],y[included],sample_weight=normalized)
            else:
                model={"neighbors":NearestNeighbors(n_neighbors=min(int(parameter),int(included.sum())),algorithm="brute").fit(XS[included]),
                       "target":y[included].copy(),"sample_weight":w[included].copy()}
            FIT_CACHE[key]=model
        estimators.append(FIT_CACHE[key])
    return {"kind":kind,"parameter":float(parameter),"scaler":classifier_scaler,"estimators":estimators,
            "features":FEATURES,"reasons":REASONS,"dataset":dataset_name,
            "logistic_weight_normalization_total":int(known.sum())}


def predict_reason_models(model, raw_features):
    scaled=model["scaler"].transform(np.asarray(raw_features))
    output=[]
    for estimator in model["estimators"]:
        if model["kind"]=="logistic":
            positive=list(estimator.classes_).index(1)
            probability=estimator.predict_proba(scaled)[:,positive]
        else:
            distance,indices=estimator["neighbors"].kneighbors(scaled)
            zero=distance==0
            exact=zero.any(axis=1)
            distance_weight=np.divide(1.,distance,out=np.zeros_like(distance),where=distance!=0)
            distance_weight[exact]=zero[exact].astype(float)
            vote=distance_weight*estimator["sample_weight"][indices]
            probability=(vote*estimator["target"][indices]).sum(axis=1)/vote.sum(axis=1)
        output.append(probability)
    probabilities=np.column_stack(output)
    assert np.isfinite(probabilities).all()
    return np.clip(probabilities,0,1)


known_y=np.full_like(original_y,-100);known_y[known]=original_y[known]
known_w=np.zeros(original_y.shape);known_w[known]=1.0
save_dataset("known_only",known_y,known_w)
legacy=pd.read_csv(DATA/"experiments/kmeans/datasets/k02_train.csv").set_index("ID")
legacy_y=known_y.copy();legacy_w=known_w.copy()
positions=pd.Index(train.ID).get_indexer(legacy.index)
assert (positions>=0).all()
legacy_y[positions]=legacy[REASONS].to_numpy(dtype=int)
legacy_w[positions]=legacy.training_weight.to_numpy()[:,None]
save_dataset("legacy_k02",legacy_y,legacy_w)
with threadpool_limits(limits=1):
    reproduced=fit_reason_models("known_only")
np.testing.assert_allclose(predict_reason_models(reproduced,X_valid),baseline.predict_proba(X_valid),rtol=1e-8,atol=1e-10)
reference_rows=[]
for name,model,kaggle in [("submitted_supervised",baseline,0.87072),("submitted_k02",legacy_packet["classifier"],0.87393)]:
    metrics=save_predictions(name,model.predict_proba(X_valid))
    reference_rows.append({"experiment":name,"dataset":"frozen reference","classifier":"logistic","parameter":1000.0,
                           "generated_trials":0 if name=="submitted_supervised" else 388,**metrics,"kaggle_score":kaggle})
print("Confirmed that the new per-reason trainer reproduces the supervised reference.")

Confirmed that the new per-reason trainer reproduces the supervised reference.


## Stage 1: compare cluster-based labeling

GMM has either diagonal component covariances or one tied covariance shared across components. Regularization 0.001 stabilizes covariance estimation in 150 dimensions. Record convergence and exclude unconverged fits. Choose the best cluster count in each family by validation sample-wise F1, with exact ties favoring smaller K. Inertia and likelihood are diagnostics, not the selection metric.

In [3]:
cluster_rows=[]
cluster_models={}
stage1_models={}


def run_clustering_candidate(family,k):
    name=f"{family}_k{k:02d}"
    with threadpool_limits(limits=1):
        if family=="kmeans":
            packet=joblib.load(DATA/"experiments/kmeans/models"/f"k{k:02d}.joblib")
            clustering=packet["kmeans"]
            ids=clustering.predict(XC)
            audit=pd.read_csv(DATA/"experiments/kmeans/label_audits"/f"k{k:02d}_labels.csv")
            assert audit.ID.tolist()==train.ID.tolist()
            np.testing.assert_array_equal(ids,audit.kmeans_cluster.to_numpy())
            responsibilities=np.eye(k)[ids]
            fit_info={"converged":True,"iterations":int(clustering.n_iter_),"inertia":float(clustering.inertia_)}
        else:
            covariance="diag" if family=="gmm_diag" else "tied"
            clustering=GaussianMixture(n_components=k,covariance_type=covariance,**GMM_RULE)
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always",ConvergenceWarning)
                clustering.fit(XC)
            fit_info={"converged":bool(clustering.converged_),"iterations":int(clustering.n_iter_),
                      "bic":float(clustering.bic(XC)),"lower_bound":float(clustering.lower_bound_)}
            if not clustering.converged_:
                cluster_rows.append({"experiment":name,"family":family,"k":k,**fit_info,"status":"excluded_unconverged"})
                print(name,"excluded because GMM did not converge.",flush=True)
                return
            responsibilities=clustering.predict_proba(XC)
        targets,weights,probabilities,support,components=partial_labels(responsibilities)
        dataset=save_dataset(name,targets,weights,probabilities,responsibilities.argmax(axis=1),support)
        components.to_csv(OUT/"clusters"/f"{name}.csv",index=False)
        model=fit_reason_models(name,"logistic",1000.0)
        metrics=save_predictions("stage1_"+name,predict_reason_models(model,X_valid))
        stage1_models[name]=model
        cluster_models[name]=clustering
        joblib.dump({"clustering":clustering,"scaler":cluster_scaler,"family":family,"k":k,"label_rule":LABEL_RULE},
                    OUT/"clustering_models"/f"{name}.joblib",compress=3)
        joblib.dump(model,OUT/"classifier_models"/f"stage1_{name}.joblib",compress=3)
        cluster_rows.append({"experiment":name,"family":family,"k":k,"generated_trials":dataset["generated_trials"],
                             **fit_info,**metrics,"status":"ok"})
        print(f"{name}: generated trials={dataset['generated_trials']}, sample F1={metrics['sample_f1']:.4f}, "
              f"macro F1={metrics['macro_f1']:.4f}",flush=True)

In [4]:
for k in K_VALUES:
    run_clustering_candidate("kmeans",k)

kmeans_k02: generated trials=0, sample F1=0.8932, macro F1=0.8043


kmeans_k05: generated trials=814, sample F1=0.8803, macro F1=0.8019


kmeans_k08: generated trials=803, sample F1=0.9017, macro F1=0.8091


kmeans_k10: generated trials=843, sample F1=0.8632, macro F1=0.7581


kmeans_k15: generated trials=988, sample F1=0.8376, macro F1=0.6808


### gmm_diag: five cluster counts

In [5]:
run_clustering_candidate("gmm_diag",2)

gmm_diag_k02: generated trials=0, sample F1=0.8932, macro F1=0.8043


In [6]:
run_clustering_candidate("gmm_diag",5)

gmm_diag_k05: generated trials=0, sample F1=0.8932, macro F1=0.8043


In [7]:
run_clustering_candidate("gmm_diag",8)

gmm_diag_k08: generated trials=665, sample F1=0.8846, macro F1=0.7896


In [8]:
run_clustering_candidate("gmm_diag",10)

gmm_diag_k10: generated trials=318, sample F1=0.8889, macro F1=0.7891


In [9]:
run_clustering_candidate("gmm_diag",15)

gmm_diag_k15: generated trials=755, sample F1=0.8761, macro F1=0.7737


### gmm_tied: five cluster counts

In [10]:
run_clustering_candidate("gmm_tied",2)

gmm_tied_k02: generated trials=0, sample F1=0.8932, macro F1=0.8043


In [11]:
run_clustering_candidate("gmm_tied",5)

gmm_tied_k05: generated trials=1014, sample F1=0.8974, macro F1=0.8053


In [12]:
run_clustering_candidate("gmm_tied",8)

gmm_tied_k08: generated trials=793, sample F1=0.8803, macro F1=0.7959


In [13]:
run_clustering_candidate("gmm_tied",10)

gmm_tied_k10: generated trials=696, sample F1=0.9060, macro F1=0.8116


In [14]:
run_clustering_candidate("gmm_tied",15)

gmm_tied_k15: generated trials=887, sample F1=0.8504, macro F1=0.7068


In [15]:
cluster_comparison=pd.DataFrame(cluster_rows)
cluster_comparison.to_csv(OUT/"clustering_comparison.csv",index=False)
label_counts=pd.DataFrame(LABEL_COUNTS)
label_counts.to_csv(OUT/"label_counts.csv",index=False)
shortlist=["known_only","legacy_k02"]
for family in ["kmeans","gmm_diag","gmm_tied"]:
    candidates=cluster_comparison.loc[(cluster_comparison.family==family)&(cluster_comparison.status=="ok")]
    if len(candidates):
        best=candidates.sort_values(["sample_f1","k"],ascending=[False,True],kind="stable").iloc[0]
        shortlist.append(best.experiment)
print("Datasets shortlisted for classifier comparison:",shortlist)
display(cluster_comparison[["experiment","generated_trials","sample_f1","macro_f1","status"]].round(4))
display(label_counts.loc[label_counts.dataset.isin(shortlist)].pivot(index="dataset",columns="reason",values="generated_positives"))

Datasets shortlisted for classifier comparison: ['known_only', 'legacy_k02', 'kmeans_k08', 'gmm_diag_k02', 'gmm_tied_k10']


,experiment,generated_trials,sample_f1,macro_f1,status
0,kmeans_k02,0,0.8932,0.8043,ok
1,kmeans_k05,814,0.8803,0.8019,ok
2,kmeans_k08,803,0.9017,0.8091,ok
3,kmeans_k10,843,0.8632,0.7581,ok
4,kmeans_k15,988,0.8376,0.6808,ok
5,gmm_diag_k02,0,0.8932,0.8043,ok
6,gmm_diag_k05,0,0.8932,0.8043,ok
7,gmm_diag_k08,665,0.8846,0.7896,ok
8,gmm_diag_k10,318,0.8889,0.7891,ok
9,gmm_diag_k15,755,0.8761,0.7737,ok


reason,Efficacy,Funding,Other,Recruitment,Safety
dataset,,,,,
gmm_diag_k02,0,0,0,0,0
gmm_tied_k10,0,0,0,608,0
kmeans_k08,0,0,0,591,0
known_only,0,0,0,0,0
legacy_k02,0,0,0,388,0


## Stage 2: compare classifiers on the shortlisted datasets

The legacy K=2 dataset is re-evaluated with the same frozen classifier scaler and normalized logistic fitting weight as every new candidate. It is therefore a controlled classifier comparison, distinct from the frozen submitted K=2 model shown as a reference.

Weighted nearest neighbors multiply inverse-distance votes by the saved training weights. Exact zero-distance matches vote alone, following the usual distance-weighting convention. Unknown labels are excluded independently for each reason. The saved model format contains only ordinary scikit-learn estimators, arrays, and dictionaries; `predict_reason_models` reconstructs probabilities.

In [16]:
classifier_rows=[]
classifier_models={}
with threadpool_limits(limits=1):
    for dataset_name in shortlist:
        for kind,parameter in CLASSIFIER_GRID:
            suffix=f"C{int(parameter)}" if kind=="logistic" else f"k{int(parameter)}"
            name=f"{dataset_name}__{kind}_{suffix}"
            model=fit_reason_models(dataset_name,kind,parameter)
            probabilities=predict_reason_models(model,X_valid)
            metrics=save_predictions(name,probabilities)
            classifier_models[name]=model
            joblib.dump(model,OUT/"classifier_models"/f"{name}.joblib",compress=3)
            classifier_rows.append({"experiment":name,"dataset":dataset_name,"classifier":kind,"parameter":parameter,
                                    "generated_trials":DATASETS[dataset_name]["generated_trials"],**metrics,"kaggle_score":np.nan})
            print(f"{name}: sample F1={metrics['sample_f1']:.4f}, macro F1={metrics['macro_f1']:.4f}",flush=True)
classifier_comparison=pd.DataFrame(classifier_rows)
classifier_comparison.to_csv(OUT/"classifier_comparison.csv",index=False)
ranking=classifier_comparison.sort_values(["sample_f1","generated_trials","experiment"],ascending=[False,True,True],kind="stable")
all_results=pd.concat([pd.DataFrame(reference_rows),classifier_comparison],ignore_index=True)
all_results.to_csv(OUT/"comparison.csv",index=False)
display(ranking.head(12)[["experiment","sample_f1","macro_f1","generated_trials"]].round(4))

known_only__logistic_C100: sample F1=0.8889, macro F1=0.7939


known_only__logistic_C1000: sample F1=0.8932, macro F1=0.8043


known_only__logistic_C10000: sample F1=0.8974, macro F1=0.8039


known_only__weighted_knn_k1: sample F1=0.8547, macro F1=0.6690


known_only__weighted_knn_k3: sample F1=0.8504, macro F1=0.5864


known_only__weighted_knn_k7: sample F1=0.8205, macro F1=0.5305


... (26 outputs left out)

gmm_tied_k10__weighted_knn_k3: sample F1=0.8526, macro F1=0.5864


gmm_tied_k10__weighted_knn_k7: sample F1=0.8376, macro F1=0.5478


gmm_tied_k10__weighted_knn_k15: sample F1=0.8462, macro F1=0.5527


,experiment,sample_f1,macro_f1,generated_trials
29,gmm_tied_k10__logistic_C1000,0.9060,0.8116,696
15,kmeans_k08__logistic_C1000,0.9017,0.8091,803
23,gmm_diag_k02__logistic_C10000,0.8974,0.8039,0
2,known_only__logistic_C10000,0.8974,0.8039,0
16,kmeans_k08__logistic_C10000,0.8974,0.8013,803
22,gmm_diag_k02__logistic_C1000,0.8932,0.8043,0
1,known_only__logistic_C1000,0.8932,0.8043,0
14,kmeans_k08__logistic_C100,0.8932,0.7801,803
21,gmm_diag_k02__logistic_C100,0.8889,0.7939,0
0,known_only__logistic_C100,0.8889,0.7939,0


In [17]:
# Controls for the weighted-neighbor implementation and saved-data replay.
with threadpool_limits(limits=1):
    native=KNeighborsClassifier(n_neighbors=7,weights="distance",algorithm="brute").fit(XS[known],original_y[known,0])
    custom=fit_reason_models("known_only","weighted_knn",7)
    np.testing.assert_allclose(predict_reason_models(custom,X_valid)[:,0],native.predict_proba(VS)[:,list(native.classes_).index(1)],rtol=1e-10,atol=1e-10)
best_overall=ranking.iloc[0]
semi_candidates=ranking.loc[ranking.generated_trials>0]
best_semi=semi_candidates.iloc[0] if len(semi_candidates) else None
selected_names=[best_overall.experiment]
if best_semi is not None and best_semi.experiment not in selected_names:selected_names.append(best_semi.experiment)
for name in selected_names:
    model=joblib.load(OUT/"classifier_models"/f"{name}.joblib")
    expected=predict_reason_models(classifier_models[name],X_valid)
    np.testing.assert_allclose(predict_reason_models(model,X_valid),expected,rtol=1e-12,atol=1e-12)
    frame=pd.read_csv(OUT/"datasets"/f"{model['dataset']}.csv",float_precision="round_trip")
    assert frame.ID.tolist()==train.ID.tolist()
    for j,r in enumerate(REASONS):
        np.testing.assert_array_equal(frame[r].to_numpy(),DATASETS[model["dataset"]]["targets"][:,j])
        np.testing.assert_allclose(frame[f"weight_{r}"].to_numpy(),DATASETS[model["dataset"]]["weights"][:,j],atol=1e-14)
    # Refit from the saved feature/target/weight table, not in-memory label generation.
    replay={**model,"estimators":[]}
    saved_scaled=model["scaler"].transform(frame[FEATURES].to_numpy())
    with threadpool_limits(limits=1):
        for j,r in enumerate(REASONS):
            y=frame[r].to_numpy(dtype=int)
            w=frame[f"weight_{r}"].to_numpy()
            included=w>0
            if model["kind"]=="logistic":
                estimator=LogisticRegression(C=model["parameter"],solver="lbfgs",max_iter=5000,tol=1e-4,random_state=SEED)
                with warnings.catch_warnings():
                    warnings.simplefilter("error",ConvergenceWarning)
                    estimator.fit(saved_scaled[included],y[included],sample_weight=w[included]*known.sum()/w[included].sum())
            else:
                estimator={"neighbors":NearestNeighbors(n_neighbors=min(int(model["parameter"]),int(included.sum())),algorithm="brute").fit(saved_scaled[included]),
                           "target":y[included],"sample_weight":w[included]}
            replay["estimators"].append(estimator)
    np.testing.assert_array_equal(label_predictions(predict_reason_models(replay,X_valid))[0],label_predictions(expected)[0])
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest

settings={
    "features":FEATURES,"extra_features_added":False,"extra_pca":False,"year_included":False,
    "k_values":K_VALUES,"label_rule":LABEL_RULE,"gmm_rule":GMM_RULE,
    "stage1_classifier":{"kind":"logistic","C":1000.0},"classifier_grid":CLASSIFIER_GRID,
    "shortlisted_datasets":shortlist,"classifier_scaler_fit_rows":157,"cluster_scaler_fit_rows":1171,
    "logistic_fitting_weight_normalization":157,"threshold":0.5,"nonempty_fallback":True,
    "validation_rows":78,"validation_used_for_fit":False,
    "best_overall":best_overall.experiment,"best_semi_supervised":None if best_semi is None else best_semi.experiment,
    "reported_kaggle_references":{"submitted_supervised":0.87072,"submitted_k02":0.87393},
    "new_kaggle_scores":None,"protected_input_sha256":protected_hashes,
    "dataset_sha256":{str(q.relative_to(OUT)):hashlib.sha256(q.read_bytes()).hexdigest() for q in sorted((OUT/"datasets").glob("*.csv"))},
    "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__},
}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
lines=["# Labeling and classifier comparison: 150 embeddings","",
       "No year, additional PCA, text, or drug features were used. All validation scores use the same 78 trials and the same 0.5 threshold with nonempty fallback.","",
       "## Frozen submitted references","",
       "| Model | Sample-wise F1 | Macro F1 | Reported Kaggle |","|---|---:|---:|---:|"]
for row in reference_rows:lines.append(f"| {row['experiment']} | {row['sample_f1']:.4f} | {row['macro_f1']:.4f} | {row['kaggle_score']:.5f} |")
lines.extend(["","## Best new classifier configurations","",
              "| Configuration | Sample-wise F1 | Macro F1 | Generated trials |","|---|---:|---:|---:|"])
for row in ranking.head(12).itertuples(index=False):lines.append(f"| {row.experiment} | {row.sample_f1:.4f} | {row.macro_f1:.4f} | {row.generated_trials} |")
lines.extend(["",f"Best new configuration: **{best_overall.experiment}**, validation sample-wise F1 **{best_overall.sample_f1:.4f}**."])
if best_semi is not None:lines.append(f"Best augmented configuration: **{best_semi.experiment}**, validation sample-wise F1 **{best_semi.sample_f1:.4f}**.")
lines.extend(["","## Clustering and generated labels","",
              "| Family | Best K | Sample-wise F1 with C=1000 | Macro F1 | Generated trials |","|---|---:|---:|---:|---:|"])
for family in ["kmeans","gmm_diag","gmm_tied"]:
    rows=cluster_comparison.loc[(cluster_comparison.family==family)&(cluster_comparison.status=="ok")].sort_values(["sample_f1","k"],ascending=[False,True])
    if len(rows):
        row=rows.iloc[0]
        lines.append(f"| {family} | {int(row.k)} | {row.sample_f1:.4f} | {row.macro_f1:.4f} | {int(row.generated_trials)} |")
lines.extend(["","For the selected augmented dataset, generated-label counts are:","",
              "| Reason | Generated positives | Generated negatives | Positive weight | Negative weight |","|---|---:|---:|---:|---:|"])
if best_semi is not None:
    for row in label_counts.loc[label_counts.dataset==best_semi.dataset].itertuples(index=False):
        lines.append(f"| {row.reason} | {row.generated_positives} | {row.generated_negatives} | {row.generated_positive_weight:.3f} | {row.generated_negative_weight:.3f} |")
lines.extend(["","## Method and limits","",
    "Stage 1 compares 15 clustering settings with the same partial-label rule and C=1000 classifier. "
    "Stage 2 shortlists one dataset per clustering family plus known-only and legacy K=2 labels, then evaluates seven classifier settings each. "
    "Some low-component labelers abstain entirely and reproduce the known-only dataset; those are not augmentation improvements. "
    "The result is the best of this staged search, not a global optimum over every method or parameter combination.","",
    "Each reason is assigned independently. Confident positives need estimated prevalence >=0.6; confident negatives need <=0.1; "
    "at least 3 effective labeled members are required. Component estimates use prior strength 2. "
    "Generated positive and negative weights are added in the same fraction of their respective known classes, capped at 0.5, "
    "with per-example weight <=0.25. If either generated class is absent, neither is used for that reason. "
    "This prevents generated negatives alone from drowning rare positive classes, without forcing equal class frequencies. "
    "Preserving known class balance is an influence-control choice, not an assertion about population prevalence.","",
    "All new classifiers use the same frozen scaler fitted on 157 known-label trials. Logistic fitting weights are normalized "
    "to a total of 157 per reason to hold effective regularization comparable. Weighted nearest-neighbor votes include both "
    "inverse distance and sample weights. Exact-distance matches vote alone. The frozen submitted K=2 model used its original "
    "scaler and weight total; its controlled re-evaluation is a separate candidate.","",
    "Unknown targets stay -100 with weight zero and are excluded independently per reason. Original known labels are preserved. "
    "No validation trial participates in fitting. Validation has been used to select among many candidates, and the new Kaggle "
    "scores remain unmeasured; small validation gains should not be presented as confirmed test improvements.","",
    "## Artifacts and verification","",
    "`datasets/` saves each exact feature/target/weight table. `label_audits/`, `clusters/`, and `label_counts.csv` expose generated labels and budgets. "
    "`clustering_models/` and `classifier_models/` save fitted models. `validation_predictions/` saves probabilities and labels. "
    "`clustering_comparison.csv`, `classifier_comparison.csv`, and `comparison.csv` retain every evaluated configuration. "
    "`settings.json` records settings, selection, versions, and checksums.","",
    "Checks reproduced baseline probabilities, matched weighted-neighbor predictions to scikit-learn when all weights equal one, "
    "preserved known labels and source artifacts, enforced partial-label/weight masks and budgets, and replayed saved selected models and data.",""])
summary="\n".join(lines).rstrip()+"\n"
(OUT/"SUMMARY.md").write_text(summary)
print(summary)

# Labeling and classifier comparison: 150 embeddings

No year, additional PCA, text, or drug features were used. All validation scores use the same 78 trials and the same 0.5 threshold with nonempty fallback.

## Frozen submitted references

| Model | Sample-wise F1 | Macro F1 | Reported Kaggle |
|---|---:|---:|---:|
| submitted_supervised | 0.8932 | 0.8043 | 0.87072 |
| submitted_k02 | 0.8568 | 0.6808 | 0.87393 |

## Best new classifier configurations

| Configuration | Sample-wise F1 | Macro F1 | Generated trials |
|---|---:|---:|---:|
| gmm_tied_k10__logistic_C1000 | 0.9060 | 0.8116 | 696 |
| kmeans_k08__logistic_C1000 | 0.9017 | 0.8091 | 803 |
| gmm_diag_k02__logistic_C10000 | 0.8974 | 0.8039 | 0 |
| known_only__logistic_C10000 | 0.8974 | 0.8039 | 0 |
| kmeans_k08__logistic_C10000 | 0.8974 | 0.8013 | 803 |
| gmm_diag_k02__logistic_C1000 | 0.8932 | 0.8043 | 0 |
| known_only__logistic_C1000 | 0.8932 | 0.8043 | 0 |
| kmeans_k08__logistic_C100 | 0.8932 | 0.7801 | 803 |
| gmm_diag_k02__

## Prepare the leading augmented candidates

Export the best revised K-means candidate and the best tied-covariance GMM candidate for manual Kaggle comparison. The fitted classifier dictionaries retain the fixed 157-row training scaler, feature order, reason order, and individual reason estimators. Use `predict_reason_models`, threshold 0.5, and the same nonempty fallback.

No candidate has a new Kaggle score yet. The current-best Git tag and reported score 0.87393 continue to refer to the previously submitted K=2 model. The new models are candidates selected by validation, not confirmed leaderboard improvements.

In [18]:
test=pd.read_csv(INPUT_DIR/"part1_test.csv")
sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
assert test.ID.is_unique and sample.ID.is_unique
assert set(test.ID)==set(sample.ID) and set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
assert np.isfinite(test[FEATURES].to_numpy()).all()
old_submission=pd.read_csv(DATA/"submissions/archive/submitted/kmeans_k2__kaggle_0.87393.csv").set_index("ID")
export_rows=[]
for prefix in ["gmm_tied_","kmeans_"]:
    candidates=ranking.loc[ranking.dataset.str.startswith(prefix)]
    if not len(candidates):continue
    selected=candidates.iloc[0]
    name=selected.experiment
    model=joblib.load(OUT/"classifier_models"/f"{name}.joblib")
    probability=predict_reason_models(model,test[FEATURES].to_numpy())
    predicted,fallback=label_predictions(probability)
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in predicted]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    assert output.Predicted.map(lambda s:set(s.split()).issubset(REASONS)).all()
    path=OUT/"submissions"/f"part1_{name.replace('__','_')}_submission.csv"
    output.to_csv(path,index=False)
    pd.testing.assert_frame_equal(pd.read_csv(path),output)
    changed=sum(set(row.Predicted.split())!=set(old_submission.loc[row.ID,"Predicted"].split()) for row in output.itertuples(index=False))
    export_rows.append({
        "experiment":name,"dataset":selected.dataset,"classifier":selected.classifier,"parameter":float(selected.parameter),
        "validation_sample_f1":float(selected.sample_f1),"validation_macro_f1":float(selected.macro_f1),
        "submission_file":str(path.relative_to(OUT)),"test_rows":len(output),"test_fallback_trials":int(fallback.sum()),
        "test_trials_changed_from_submitted_k02":int(changed),
        "submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
        "model_sha256":hashlib.sha256((OUT/"classifier_models"/f"{name}.joblib").read_bytes()).hexdigest(),
        "kaggle_score":None,"kaggle_status":"pending manual submission",
    })
exports=pd.DataFrame(export_rows)
exports.to_csv(OUT/"submissions.csv",index=False)
(OUT/"submission_manifest.json").write_text(json.dumps({"submissions":export_rows,"comparator_score":0.87393,
    "comparator_source":"reported by Michael for original K=2; current-best-kaggle tag unchanged"},indent=2)+"\n")
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest

summary_path=OUT/"SUMMARY.md"
text=summary_path.read_text().split("## Candidate Kaggle files")[0].rstrip()
export_lines=["","","## Candidate Kaggle files","",
              "These are local files for manual upload; neither candidate has a measured Kaggle score. Compare against **0.87393**.","",
              "| Candidate | Validation F1 | Submission file | Changed test trials from original K=2 |",
              "|---|---:|---|---:|"]
for row in export_rows:
    export_lines.append(f"| {row['experiment']} | {row['validation_sample_f1']:.4f} | `{row['submission_file']}` | {row['test_trials_changed_from_submitted_k02']} |")
summary_path.write_text((text+"\n".join(export_lines)).rstrip()+"\n")
display(exports.drop(columns=["submission_sha256","model_sha256"]).round(4))
print("Candidate files saved under:",OUT/"submissions")

,experiment,dataset,classifier,parameter,validation_sample_f1,validation_macro_f1,submission_file,test_rows,test_fallback_trials,test_trials_changed_from_submitted_k02,kaggle_score,kaggle_status
0,gmm_tied_k10__logistic_C1000,gmm_tied_k10,logistic,1000.0,0.9060,0.8116,submissions/part1_gmm_tied_k10_logistic_C1000_...,312,14,54,None,pending manual submission
1,kmeans_k08__logistic_C1000,kmeans_k08,logistic,1000.0,0.9017,0.8091,submissions/part1_kmeans_k08_logistic_C1000_su...,312,12,58,None,pending manual submission


Candidate files saved under: /Users/michael/Documents/School/Cornell/AppliedMachineLearning/AML-Midterm/part1/experiments/method_comparison/submissions
